# IND320 Assignment 4

## Introduction

This is a Jupyter Notebook for the fourth part of the IND320 - Data to Decision Project Work. This notebook includes:

* Links to Github repository and Streamlit app
* Description of AI Usage
* NOK/USD and NOK/EUR currency exchange rates from Norges Bank API, Open-Meteo weather data for NVE areas, SPC and LOF outlier detection, STL, a spectrogram, and locations saved for the app.

### Video and Links to Github and Streamlit

Github: https://github.com/MagnusFrantsen/IND320-MagnusFrantsen  
Streamlit: https://ind320-magnusfrantsen.streamlit.app/

The video is uploaded to the Github Repository as: "IND320_Project_Work_Part_4.mp4"

### Log Describing Compulsory Work

This section contains a log of the workflow with the Jupyter Notebook and Streamlit App:

I continued on the demo branch in the same Github Repository and Streamlit app, to make sure my changes did not affect the feedback for part 3.

First I inspected the Norges Bank API on "https://www.norges-bank.no/en/topics/statistics/open-data/guide-data-warehouse/" to figure out how to build up the request to make the API call.



### AI Usage

The AI model used during this assignment was Claude Opus 5.5 and Claude Fable 5.2 (the few days it was a part of the Pro-plan).

Generally, the AI model has suggested code, and most of the code in this assignment was copied from these suggestions and then adapted by me. I have tried to understand each part by asking follow-up questions about what the code does and why it is needed.

More specifically the AI model has been used for:

* Breaking down the task description into concrete steps for both the notebook and the Streamlit app
* Finding the correct layers in NVE Atlas and checking the downloaded shapefiles
* Clipping the areas to land and sampling 1000 random locations in each area
* Creating the Open-Meteo download function, and reducing the number of API calls by merging locations into ERA5 grid cells
* Creating the functions for the DCT filtering, SPC boundaries, LOF, STL and spectrogram, and the tests of these functions
* Explaining concepts such as the Discrete Cosine Transform, robust statistics and the use of st.stop()
* Designing the global area selector and how the weather data is loaded in the Streamlit app
* Debugging errors, and finding that the VASS areas in the data did not match the areas listed by the API
* Restructuring the Streamlit pages, the navigation and the home page, and providing documentation for st.tabs and st.navigation
* Writing a first draft of the work log and this section, which I reviewed and edited

I have personally reviewed, tested and adapted the generated code, and written most of the comments myself, to make sure I understand what the code is actually doing and to get the desired functionality.

# Tasks

## Downloading Euro and USD currency exchange rates

### Importing needed modules

In [10]:
import requests
import pandas as pd
import io

### Defining the base URL of the API

In [2]:
BASE_URL = "https://data.norges-bank.no/api/data/EXR/"

### Defining the download function

In [14]:
def get_currency_exchange_rates(currencies, start, end, freq="B"):
    """Fetching NOK exchange rates from NB as a pandas DataFrame

    currencies: list of ISO code, here ["USD", "EUR"]
    start, end: dates in the format "YYYY-MM-DD"
    freq: "B" (business daily), "M" (monthly) and "A" (annual)
    """

    key = f"{freq}.{'+'.join(currencies)}.NOK.SP"

    params = {
        "format": "csv",       # one observation per row
        "startPeriod": start,
        "endPeriod": end,
        "locale": "en",
    }

    response = requests.get(BASE_URL + key, params=params, timeout=30)

    response.raise_for_status()

    # Separating the csv-formatted files by semicolon
    exchange_rates = pd.read_csv(io.StringIO(response.text), sep=";")

    # Converting the time-column to datetime timestamps
    exchange_rates["TIME_PERIOD"] = pd.to_datetime(exchange_rates["TIME_PERIOD"])

    # Returning a pivot table from the exchange_rates DataFrame 
    return exchange_rates.pivot(index="TIME_PERIOD", columns = "BASE_CUR", values = "OBS_VALUE")

### Downloading the data

In [ ]:
# Verifying that the function is able to download the data from 2024-01-01 to 2026-11-13
exchange_rates = get_currency_exchange_rates(["USD", "EUR"], "2024-01-01", "2026-11-13")
# Printing the last 5 rows of the pivot
print(exchange_rates.tail())

BASE_CUR         EUR     USD
TIME_PERIOD                 
2026-11-09   10.5325  9.4099
2026-11-10   10.6155  9.4663
2026-11-11   10.6145  9.5394
2026-11-12   10.5630  9.5325
2026-11-13   10.5695  9.5634


## Time Zone Differences

Below is a table used to summarize the Time Zone Convention used by the different APIs as default, also including the resolution of the data. 

| Source            | Resolution       | Time convention            |
|-------------------|------------------|----------------------------|
| Magasinstatistikk | Weekly           | Date only, Norwegian time  |
| ENTSO-E           | Hourly           | UTC                        |
| Open-Meteo        | Hourly           | GMT by default             |
| Norges Bank       | Business daily   | Date only                  |

Here is how the APIs are treated at this point in the assignments:

- **ENTSO-E:** requested, parsed and stored in UTC; Spark session time zone set to UTC.
- **Open-Meteo:** requested with `timezone="UTC"` and parsed as UTC.
- **Magasinstatistikk and Norges Bank:** date only, treated as Norwegian calendar dates; no conversion needed.
- **Grouping and plotting:** UTC timestamps are converted to `Europe/Oslo` before grouping by year and month.
- **Combining sources:** hourly data is converted to `Europe/Oslo` and aggregated to Monday–Sunday weeks to match the weekly reservoir data.

As I am going to make a forecasting model, I need one row per time step. Because the lowest resolution is weekly (Magasinstatistikk), I have to convert the other data to a weekly resolution, because that will be the highest resolution the model will be able to predict at, since the data spans several years, I think a weekly or even monthly resolution will be sufficient. And I will test both below.